In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/641.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/173.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/815.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/491.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/718.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/709.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/379.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/780.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/248.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/94.png
/kaggle/inp

In [2]:
# Kaggle environment boilerplate
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/641.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/173.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/815.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/491.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/718.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/709.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/379.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/780.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/248.png
/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data/low_res/94.png
/kaggle/inp

In [3]:
import os, gc, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from torchvision.models import vgg19, VGG19_Weights
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

device  = "cuda" if torch.cuda.is_available() else "cpu"
NUM_GPUS = torch.cuda.device_count()
print(f"Device : {device}  |  GPUs: {NUM_GPUS}")
if device == "cuda":
    for i in range(NUM_GPUS):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

Device : cuda  |  GPUs: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4


In [4]:
CFG = dict(
    data_root    = "/kaggle/input/datasets/adityachandrasekhar/image-super-resolution/dataset/Raw Data",
    lr_size      = 32,      # low-res input
    hr_size      = 128,     # high-res target  (4x upscale)
    batch_size   = 32,
    num_epochs   = 70,
    lr_G         = 1e-4,
    lr_D         = 1e-4,
    n_res_blocks = 8,       # ResBlocks in SRGAN
    n_rrdb       = 6,       # RRDB blocks in ESRGAN
    lambda_perc  = 1e-2,    # perceptual loss weight
    lambda_adv   = 1e-3,    # adversarial loss weight
    vis_every    = 5,       # save sample grid every N epochs
    save_dir     = "/kaggle/working/outputs",
)
os.makedirs(CFG["save_dir"], exist_ok=True)
print("Config ready. Output dir:", CFG["save_dir"])

Config ready. Output dir: /kaggle/working/outputs


In [5]:
class SRDataset(Dataset):
    """Paired LR / HR super-resolution dataset."""
    def __init__(self, root_dir, lr_size=32, hr_size=128):
        self.lr_path   = os.path.join(root_dir, "low_res")
        self.hr_path   = os.path.join(root_dir, "high_res")
        self.lr_images = sorted(os.listdir(self.lr_path))
        self.hr_images = sorted(os.listdir(self.hr_path))
        assert len(self.lr_images) == len(self.hr_images), "LR/HR count mismatch"

        self.lr_tf = transforms.Compose([
            transforms.Resize((lr_size, lr_size),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),   # → [-1, 1]
        ])
        self.hr_tf = transforms.Compose([
            transforms.Resize((hr_size, hr_size),
                              interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self):
        return len(self.lr_images)

    def __getitem__(self, idx):
        lr = Image.open(os.path.join(self.lr_path, self.lr_images[idx])).convert("RGB")
        hr = Image.open(os.path.join(self.hr_path, self.hr_images[idx])).convert("RGB")
        return self.lr_tf(lr), self.hr_tf(hr)


def denorm(t):
    """[-1, 1] → [0, 1] for display / metrics."""
    return (t * 0.5 + 0.5).clamp(0, 1)


dataset    = SRDataset(CFG["data_root"], CFG["lr_size"], CFG["hr_size"])
dataloader = DataLoader(dataset, batch_size=CFG["batch_size"],
                        shuffle=True, num_workers=2, pin_memory=True)
print(f"Dataset : {len(dataset)} images  |  Batches/epoch : {len(dataloader)}")

Dataset : 855 images  |  Batches/epoch : 27


In [6]:
vgg_feat = vgg19(weights=VGG19_Weights.DEFAULT).features[:36].to(device).eval()
for p in vgg_feat.parameters():
    p.requires_grad = False
if NUM_GPUS > 1:
    vgg_feat = nn.DataParallel(vgg_feat)

# ImageNet normalisation constants
vgg_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
vgg_std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def perceptual_loss(fake, real):
    """VGG feature-space MSE loss (expects [-1,1] input)."""
    fake_n = (denorm(fake) - vgg_mean) / vgg_std
    real_n = (denorm(real) - vgg_mean) / vgg_std
    return nn.functional.mse_loss(vgg_feat(fake_n), vgg_feat(real_n))

print("VGG-19 perceptual network ready.")

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:03<00:00, 171MB/s]


VGG-19 perceptual network ready.


In [7]:
# ─── GAN Generator (pixel-loss baseline) ──────────────────────────────────────
class GAN_G(nn.Module):
    """Lightweight sub-pixel upsampling baseline."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,  64,  9, padding=4), nn.ReLU(inplace=True),
            nn.Conv2d(64, 256, 3, padding=1), nn.PixelShuffle(2), nn.ReLU(inplace=True),
            nn.Conv2d(64, 256, 3, padding=1), nn.PixelShuffle(2), nn.ReLU(inplace=True),
            nn.Conv2d(64, 3,   9, padding=4), nn.Tanh(),
        )
    def forward(self, x):
        return self.net(x)


# ─── SRGAN Generator ───────────────────────────────────────────────────────────
class ResBlock(nn.Module):
    """Standard residual block with BatchNorm (SRGAN style)."""
    def __init__(self, c=64):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(c, c, 3, 1, 1), nn.BatchNorm2d(c), nn.PReLU(),
            nn.Conv2d(c, c, 3, 1, 1), nn.BatchNorm2d(c),
        )
    def forward(self, x):
        return x + self.block(x)

class SRGAN_G(nn.Module):
    """SRGAN generator: ResNet backbone + sub-pixel upsampling."""
    def __init__(self, n_res=8):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 9, 1, 4)
        self.prelu = nn.PReLU()
        self.res   = nn.Sequential(*[ResBlock(64) for _ in range(n_res)])
        self.conv2 = nn.Sequential(nn.Conv2d(64, 64, 3, 1, 1), nn.BatchNorm2d(64))
        self.up    = nn.Sequential(
            nn.Conv2d(64, 256, 3, 1, 1), nn.PixelShuffle(2), nn.PReLU(),
            nn.Conv2d(64, 256, 3, 1, 1), nn.PixelShuffle(2), nn.PReLU(),
        )
        self.out   = nn.Sequential(nn.Conv2d(64, 3, 9, 1, 4), nn.Tanh())

    def forward(self, x):
        x1 = self.prelu(self.conv1(x))
        x  = self.conv2(self.res(x1)) + x1
        return self.out(self.up(x))


# ─── ESRGAN Generator (Dense RRDB) ─────────────────────────────────────────────
class DenseLayer(nn.Module):
    def __init__(self, in_c, gc=32):
        super().__init__()
        self.conv  = nn.Conv2d(in_c, gc, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
    def forward(self, x):
        return self.lrelu(self.conv(x))

class RRDB(nn.Module):
    """Residual-in-Residual Dense Block."""
    def __init__(self, c=64, gc=32):
        super().__init__()
        self.d1   = DenseLayer(c,      gc)
        self.d2   = DenseLayer(c+gc,   gc)
        self.d3   = DenseLayer(c+2*gc, gc)
        self.fuse = nn.Conv2d(c+3*gc, c, 1)
    def forward(self, x):
        d1 = self.d1(x)
        d2 = self.d2(torch.cat([x, d1], 1))
        d3 = self.d3(torch.cat([x, d1, d2], 1))
        return x + 0.2 * self.fuse(torch.cat([x, d1, d2, d3], 1))

class ESRGAN_G(nn.Module):
    """ESRGAN generator: RRDB trunk + sub-pixel upsampling."""
    def __init__(self, n_rrdb=6):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1)
        self.rrdb  = nn.Sequential(*[RRDB(64) for _ in range(n_rrdb)])
        self.conv2 = nn.Conv2d(64, 64, 3, 1, 1)
        self.up    = nn.Sequential(
            nn.Conv2d(64, 256, 3, 1, 1), nn.PixelShuffle(2), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 256, 3, 1, 1), nn.PixelShuffle(2), nn.LeakyReLU(0.2, True),
        )
        self.out   = nn.Sequential(
            nn.Conv2d(64, 64, 3, 1, 1), nn.LeakyReLU(0.2, True),
            nn.Conv2d(64, 3,  3, 1, 1), nn.Tanh(),
        )
    def forward(self, x):
        x1 = self.conv1(x)
        x  = self.conv2(self.rrdb(x1)) + x1
        return self.out(self.up(x))


# ─── PatchGAN Discriminator (shared) ───────────────────────────────────────────
class Discriminator(nn.Module):
    """PatchGAN: output is a spatial map of real/fake scores."""
    def __init__(self):
        super().__init__()
        def blk(ci, co, stride):
            layers = [nn.Conv2d(ci, co, 4, stride, 1), nn.LeakyReLU(0.2, True)]
            if co > 64:
                layers.insert(1, nn.InstanceNorm2d(co))
            return layers
        self.model = nn.Sequential(
            *blk(3,   64,  2),
            *blk(64,  128, 2),
            *blk(128, 256, 2),
            *blk(256, 512, 1),
            nn.Conv2d(512, 1, 4, 1, 1)   # PatchGAN output
        )
    def forward(self, x):
        return self.model(x)


def wrap(model):
    """Move to device and optionally wrap in DataParallel."""
    model = model.to(device)
    return nn.DataParallel(model) if NUM_GPUS > 1 else model

print("All architectures defined:")
for cls in [GAN_G, SRGAN_G, ESRGAN_G, Discriminator]:
    n = sum(p.numel() for p in cls().parameters()) / 1e6
    print(f"  {cls.__name__:<15} {n:.2f} M params")

All architectures defined:
  GAN_G           0.33 M params
  SRGAN_G         0.96 M params
  ESRGAN_G        0.93 M params
  Discriminator   2.76 M params


In [8]:
psnr_fn = PeakSignalNoiseRatio(data_range=1.0).to(device)
ssim_fn = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)


def compute_metrics(G, loader):
    """PSNR + SSIM averaged over the full loader."""
    G.eval()
    total_psnr = total_ssim = 0.0
    n = 0
    with torch.no_grad():
        for lr_b, hr_b in loader:
            lr_b = lr_b.to(device)
            hr_b = hr_b.to(device)
            sr   = denorm(G(lr_b))
            hr01 = denorm(hr_b)
            total_psnr += psnr_fn(sr, hr01).item()
            total_ssim += ssim_fn(sr, hr01).item()
            n += 1
    return total_psnr / n, total_ssim / n


def save_sample_grid(G, fix_lr, fix_hr, model_name, epoch):
    """Save an LR / SR / HR comparison grid for a fixed sample."""
    G.eval()
    with torch.no_grad():
        sr = denorm(G(fix_lr)).cpu()
    lr_show = denorm(fix_lr).cpu()
    hr_show = denorm(fix_hr).cpu()

    n = min(4, sr.size(0))
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    for i in range(n):
        for j, (img, lbl) in enumerate([
            (lr_show[i], "LR Input"),
            (sr[i],      "SR Output"),
            (hr_show[i], "HR Target"),
        ]):
            ax = axes[i, j] if n > 1 else axes[j]
            ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy())
            if i == 0:
                ax.set_title(lbl, fontsize=10, weight="bold")
            ax.axis("off")

    plt.suptitle(f"{model_name} — Epoch {epoch}", fontsize=12, weight="bold")
    plt.tight_layout()
    out = os.path.join(CFG["save_dir"], f"{model_name}_ep{epoch:03d}.png")
    plt.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"  ↳ sample grid saved → {out}")


print("Helper functions ready.")

Helper functions ready.


In [9]:
def train_model(G_class, model_name, epochs=CFG["num_epochs"]):
    """
    Train a GAN generator with the appropriate loss strategy:
      - GAN    : pixel (L1) loss only
      - SRGAN  : pixel + perceptual + adversarial
      - ESRGAN : perceptual + adversarial + 0.5 × pixel
    Returns (trained_G, history_dict).
    """
    print(f"\n{'='*60}")
    print(f"  Training {model_name}  ({epochs} epochs)")
    print(f"{'='*60}")

    G = wrap(G_class())
    D = wrap(Discriminator())

    opt_G   = optim.Adam(G.parameters(), lr=CFG["lr_G"], betas=(0.9, 0.999))
    opt_D   = optim.Adam(D.parameters(), lr=CFG["lr_D"], betas=(0.9, 0.999))
    sched_G = optim.lr_scheduler.CosineAnnealingLR(opt_G, epochs)
    sched_D = optim.lr_scheduler.CosineAnnealingLR(opt_D, epochs)

    bce = nn.BCEWithLogitsLoss()
    l1  = nn.L1Loss()

    history = {"g_loss": [], "d_loss": [], "psnr": [], "ssim": []}
    best_psnr = 0.0

    # Fixed sample for consistent in-training visualisation
    fix_lr, fix_hr = next(iter(dataloader))
    fix_lr = fix_lr[:8].to(device)
    fix_hr = fix_hr[:8].to(device)

    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        g_run = d_run = 0.0
        t0 = time.time()

        for lr_b, hr_b in tqdm(dataloader,
                               desc=f"[{model_name}] Ep {epoch:02d}/{epochs}",
                               leave=False):
            lr_b = lr_b.to(device, non_blocking=True)
            hr_b = hr_b.to(device, non_blocking=True)

            # Label smoothing (real=0.9, fake=0.0)
            real_lbl = torch.full(D(hr_b).shape, 0.9, device=device)
            fake_lbl = torch.zeros_like(real_lbl)

            # ── Discriminator step ──────────────────────────────────
            with torch.no_grad():
                fake_hr_d = G(lr_b)
            d_loss = 0.5 * (bce(D(hr_b),      real_lbl) +
                             bce(D(fake_hr_d), fake_lbl))
            opt_D.zero_grad(set_to_none=True)
            d_loss.backward()
            opt_D.step()

            # ── Generator step ──────────────────────────────────────
            fake_hr = G(lr_b)
            adv     = bce(D(fake_hr), real_lbl)
            pix     = l1(fake_hr, hr_b)

            if model_name == "GAN":
                g_loss = pix
            elif model_name == "SRGAN":
                g_loss = (pix
                          + CFG["lambda_perc"] * perceptual_loss(fake_hr, hr_b)
                          + CFG["lambda_adv"]  * adv)
            else:  # ESRGAN
                g_loss = (CFG["lambda_perc"] * perceptual_loss(fake_hr, hr_b)
                          + CFG["lambda_adv"]  * adv
                          + 0.5              * pix)

            opt_G.zero_grad(set_to_none=True)
            g_loss.backward()
            opt_G.step()

            g_run += g_loss.item()
            d_run += d_loss.item()

        sched_G.step()
        sched_D.step()

        # ── Epoch-level metrics ────────────────────────────────────
        ep_psnr, ep_ssim = compute_metrics(G, dataloader)
        g_avg = g_run / len(dataloader)
        d_avg = d_run / len(dataloader)

        history["g_loss"].append(g_avg)
        history["d_loss"].append(d_avg)
        history["psnr"  ].append(ep_psnr)
        history["ssim"  ].append(ep_ssim)

        elapsed = time.time() - t0
        print(f"  Ep {epoch:03d}/{epochs}  "
              f"G={g_avg:.4f}  D={d_avg:.4f}  "
              f"PSNR={ep_psnr:.2f} dB  SSIM={ep_ssim:.4f}  "
              f"({elapsed:.1f}s)")

        # ── Live visualisation ─────────────────────────────────────
        if epoch % CFG["vis_every"] == 0 or epoch == epochs:
            save_sample_grid(G, fix_lr, fix_hr, model_name, epoch)

        # ── Save best checkpoint ───────────────────────────────────
        if ep_psnr > best_psnr:
            best_psnr = ep_psnr
            path = os.path.join(CFG["save_dir"], f"{model_name}_best.pth")
            torch.save(G.state_dict(), path)

    print(f"\n  ✓ {model_name} done. Best PSNR: {best_psnr:.3f} dB")
    return G, history

In [10]:
gan_model,   gan_hist   = train_model(GAN_G,    "GAN")
srgan_model,  srgan_hist  = train_model(SRGAN_G,  "SRGAN")
esrgan_model, esrgan_hist = train_model(ESRGAN_G, "ESRGAN")

all_models = {"GAN": gan_model, "SRGAN": srgan_model, "ESRGAN": esrgan_model}
all_hist   = {"GAN": gan_hist,  "SRGAN": srgan_hist,  "ESRGAN": esrgan_hist}
print("\nAll three models trained successfully!")


  Training GAN  (70 epochs)


[GAN] Ep 01/70:   0%|          | 0/27 [00:00<?, ?it/s]

  Ep 001/70  G=0.4125  D=0.3495  PSNR=12.95 dB  SSIM=0.3215  (15.8s)


[GAN] Ep 02/70:   0%|          | 0/27 [00:00<?, ?it/s]

  Ep 002/70  G=0.3058  D=0.2182  PSNR=15.35 dB  SSIM=0.3484  (10.8s)


...

  Ep 068/70  G=0.0580  D=0.1687  PSNR=22.51 dB  SSIM=0.6419  (15.8s)


[ESRGAN] Ep 69/70:   0%|          | 0/27 [00:00<?, ?it/s]

  Ep 069/70  G=0.0579  D=0.1687  PSNR=22.51 dB  SSIM=0.6419  (15.9s)


[ESRGAN] Ep 70/70:   0%|          | 0/27 [00:00<?, ?it/s]

  Ep 070/70  G=0.0580  D=0.1687  PSNR=22.52 dB  SSIM=0.6419  (15.8s)
  ↳ sample grid saved → /kaggle/working/outputs/ESRGAN_ep070.png

  ✓ ESRGAN done. Best PSNR: 22.922 dB

All three models trained successfully!


In [11]:
COLORS = {"GAN": "#e74c3c", "SRGAN": "#f39c12", "ESRGAN": "#2980b9"}
epochs  = range(1, CFG["num_epochs"] + 1)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle("Training Curves — GAN vs SRGAN vs ESRGAN",
             fontsize=16, weight="bold", y=1.01)

metrics_cfg = [
    ("g_loss", "Generator Loss",     axes[0, 0], False),
    ("d_loss", "Discriminator Loss", axes[0, 1], False),
    ("psnr",   "PSNR (dB) ↑",        axes[1, 0], True),
    ("ssim",   "SSIM ↑",             axes[1, 1], True),
]

for key, ylabel, ax, mark_best in metrics_cfg:
    for name, h in all_hist.items():
        vals = h[key]
        ax.plot(epochs, vals, label=name, color=COLORS[name],
                linewidth=2.2, alpha=0.9)
        if mark_best:
            best_ep = int(np.argmax(vals)) + 1
            ax.axvline(best_ep, color=COLORS[name],
                       linestyle="--", linewidth=0.8, alpha=0.5)
            ax.scatter(best_ep, vals[best_ep - 1], color=COLORS[name],
                       zorder=5, s=60)

    ax.set_xlabel("Epoch", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(ylabel, fontsize=12, weight="bold")
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.set_xlim(1, CFG["num_epochs"])

plt.tight_layout()
out = os.path.join(CFG["save_dir"], "training_curves.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

Saved → /kaggle/working/outputs/training_curves.png


In [12]:
# Bicubic baseline
def bicubic_metrics():
    upsamp = nn.Upsample(size=(CFG["hr_size"], CFG["hr_size"]),
                         mode="bicubic", align_corners=False).to(device)
    bp = bs = 0.0; n = 0
    with torch.no_grad():
        for lr_b, hr_b in dataloader:
            lr_b = lr_b.to(device); hr_b = hr_b.to(device)
            bc   = denorm(upsamp(lr_b)).clamp(0, 1)
            hr01 = denorm(hr_b)
            bp += psnr_fn(bc, hr01).item()
            bs += ssim_fn(bc, hr01).item()
            n  += 1
    return bp / n, bs / n

bc_p, bc_s = bicubic_metrics()

# GAN models
results = {"Bicubic": {"psnr": bc_p, "ssim": bc_s}}
print(f"{'Model':<12}  {'PSNR (dB)':>10}  {'SSIM':>8}")
print("-" * 35)
print(f"{'Bicubic':<12}  {bc_p:>10.3f}  {bc_s:>8.4f}  (no-learning baseline)")

for name, model in all_models.items():
    p, s = compute_metrics(model, dataloader)
    results[name] = {"psnr": p, "ssim": s}
    best_ep = int(np.argmax(all_hist[name]["psnr"])) + 1
    best_v  = max(all_hist[name]["psnr"])
    print(f"{name:<12}  {p:>10.3f}  {s:>8.4f}  (best ep: {best_ep}, peak {best_v:.3f})")

winner = max({k: v for k, v in results.items() if k != "Bicubic"},
             key=lambda k: results[k]["psnr"])
print(f"\n🏆  Best model: {winner}  "
      f"PSNR={results[winner]['psnr']:.3f} dB  "
      f"SSIM={results[winner]['ssim']:.4f}")

Model          PSNR (dB)      SSIM
-----------------------------------
Bicubic           22.764    0.6842  (no-learning baseline)
GAN               21.708    0.6049  (best ep: 70, peak 21.714)
SRGAN             22.765    0.6591  (best ep: 70, peak 22.758)
ESRGAN            22.514    0.6416  (best ep: 38, peak 22.922)

🏆  Best model: SRGAN  PSNR=22.765 dB  SSIM=0.6591


In [13]:
bar_colors = ["#95a5a6", "#e74c3c", "#f39c12", "#2980b9"]
names  = list(results.keys())         # Bicubic, GAN, SRGAN, ESRGAN
psnrs  = [results[n]["psnr"] for n in names]
ssims  = [results[n]["ssim"] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle("Final Model Comparison (Full Dataset)", fontsize=15, weight="bold")

x = np.arange(len(names))
width = 0.55

bars1 = ax1.bar(x, psnrs, width, color=bar_colors, edgecolor="black", linewidth=0.8)
ax1.set_xticks(x); ax1.set_xticklabels(names, fontsize=12)
ax1.set_ylabel("PSNR (dB)", fontsize=12)
ax1.set_title("Peak Signal-to-Noise Ratio ↑", fontsize=13, weight="bold")
ax1.set_ylim(min(psnrs) * 0.97, max(psnrs) * 1.03)
ax1.grid(axis="y", alpha=0.3, linestyle="--")
for bar, v in zip(bars1, psnrs):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
             f"{v:.2f}", ha="center", va="bottom", fontsize=12, weight="bold")

bars2 = ax2.bar(x, ssims, width, color=bar_colors, edgecolor="black", linewidth=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(names, fontsize=12)
ax2.set_ylabel("SSIM", fontsize=12)
ax2.set_title("Structural Similarity Index ↑", fontsize=13, weight="bold")
ax2.set_ylim(min(ssims) * 0.97, max(ssims) * 1.01)
ax2.grid(axis="y", alpha=0.3, linestyle="--")
for bar, v in zip(bars2, ssims):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
             f"{v:.4f}", ha="center", va="bottom", fontsize=12, weight="bold")

# Highlight winner
best_idx = names.index(winner)
for ax, bars in [(ax1, bars1), (ax2, bars2)]:
    bars[best_idx].set_edgecolor("gold")
    bars[best_idx].set_linewidth(3)

plt.tight_layout()
out = os.path.join(CFG["save_dir"], "metric_bars.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

Saved → /kaggle/working/outputs/metric_bars.png


In [14]:
def plot_visual_comparison(n_samples=4):
    fix_lr, fix_hr = next(iter(dataloader))
    fix_lr = fix_lr[:n_samples].to(device)
    fix_hr = fix_hr[:n_samples].to(device)

    col_labels = ["LR  (32×32)", "GAN", "SRGAN", "ESRGAN", "HR  (128×128)"]
    n_cols = len(col_labels)

    fig = plt.figure(figsize=(n_cols * 3.2, n_samples * 3.2))
    gs  = gridspec.GridSpec(n_samples + 1, n_cols,
                            height_ratios=[0.25] + [1] * n_samples,
                            hspace=0.05, wspace=0.05)

    # Column headers
    header_colors = ["#7f8c8d", "#e74c3c", "#f39c12", "#2980b9", "#27ae60"]
    for col, (lbl, hc) in enumerate(zip(col_labels, header_colors)):
        ax = fig.add_subplot(gs[0, col])
        ax.text(0.5, 0.5, lbl, ha="center", va="center",
                fontsize=11, weight="bold", color="white",
                transform=ax.transAxes)
        ax.set_facecolor(hc); ax.axis("off")

    for row in range(n_samples):
        imgs = [denorm(fix_lr[row]).cpu()]
        for name, G in all_models.items():
            G.eval()
            with torch.no_grad():
                sr = denorm(G(fix_lr[row:row+1]))[0].cpu()
            imgs.append(sr)
        imgs.append(denorm(fix_hr[row]).cpu())

        for col, img in enumerate(imgs):
            ax = fig.add_subplot(gs[row + 1, col])
            ax.imshow(img.permute(1, 2, 0).clamp(0, 1).numpy(),
                      interpolation="nearest")
            ax.axis("off")

    plt.suptitle("Visual Comparison: LR  →  GAN / SRGAN / ESRGAN  →  HR",
                 fontsize=14, weight="bold", y=1.005)
    out = os.path.join(CFG["save_dir"], "visual_comparison.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out}")

plot_visual_comparison(n_samples=4)

Saved → /kaggle/working/outputs/visual_comparison.png


In [15]:
import matplotlib.patches as mpatches

# Normalise each metric to [0,1] range across models (excl. Bicubic)
model_names = ["GAN", "SRGAN", "ESRGAN"]
metrics_radar = {
    "PSNR":  [results[n]["psnr"]  for n in model_names],
    "SSIM":  [results[n]["ssim"]  for n in model_names],
    "G-Loss
(inv)": [1 / (1 + all_hist[n]["g_loss"][-1]) for n in model_names],
    "D-Loss
(inv)": [1 / (1 + all_hist[n]["d_loss"][-1]) for n in model_names],
}

# Min-max normalise each spoke
def minmax(vals):
    lo, hi = min(vals), max(vals)
    return [(v - lo) / (hi - lo + 1e-9) for v in vals]

spokes    = list(metrics_radar.keys())
N         = len(spokes)
angles    = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles   += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(spokes, fontsize=11)

radar_colors = {"GAN": "#e74c3c", "SRGAN": "#f39c12", "ESRGAN": "#2980b9"}
for name in model_names:
    vals_norm = [minmax(metrics_radar[s])[model_names.index(name)]
                 for s in spokes]
    vals_norm += vals_norm[:1]
    ax.plot(angles, vals_norm, color=radar_colors[name], linewidth=2.2, label=name)
    ax.fill(angles, vals_norm, color=radar_colors[name], alpha=0.15)

ax.set_yticklabels([])
ax.set_title("Normalised Performance Radar", size=13, weight="bold", pad=18)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), fontsize=11)

plt.tight_layout()
out = os.path.join(CFG["save_dir"], "radar_chart.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

SyntaxError: unterminated string literal (detected at line 8) (3967886580.py, line 8)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for name, h in all_hist.items():
    best_so_far = np.maximum.accumulate(h["psnr"])
    ax.plot(epochs, best_so_far, label=name, color=COLORS[name],
            linewidth=2.5, linestyle="-")
    ax.plot(epochs, h["psnr"], color=COLORS[name],
            linewidth=1, linestyle=":", alpha=0.5)

# Bicubic flat line
ax.axhline(bc_p, color="#95a5a6", linewidth=1.5,
           linestyle="--", label=f"Bicubic ({bc_p:.2f} dB)")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("PSNR (dB)", fontsize=12)
ax.set_title("Cumulative Best PSNR over Training  (solid = best-so-far, dotted = epoch)",
             fontsize=13, weight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, linestyle="--")
ax.set_xlim(1, CFG["num_epochs"])

plt.tight_layout()
out = os.path.join(CFG["save_dir"], "psnr_progression.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out}")

In [ ]:
print("\n" + "═"*65)
print(f"{'Model':<12} {'PSNR (dB)':>10} {'SSIM':>8} {'G-Loss':>10} {'Gain vs Bicubic':>17}")
print("─"*65)
print(f"{'Bicubic':<12} {bc_p:>10.3f} {bc_s:>8.4f}  {'—':>10}  {'—':>15}")
for name in ["GAN", "SRGAN", "ESRGAN"]:
    r  = results[name]
    gl = all_hist[name]["g_loss"][-1]
    gain_p = r["psnr"] - bc_p
    gain_s = r["ssim"] - bc_s
    print(f"{name:<12} {r['psnr']:>10.3f} {r['ssim']:>8.4f} {gl:>10.4f}"
          f"  +{gain_p:.2f} dB / +{gain_s:.4f}")
print("═"*65)

winner = max(["GAN","SRGAN","ESRGAN"], key=lambda k: results[k]["psnr"])
print(f"\n🏆  Best model : {winner}")
print(f"    PSNR       : {results[winner]['psnr']:.3f} dB")
print(f"    SSIM       : {results[winner]['ssim']:.4f}")
print(f"\nAll outputs saved to: {CFG['save_dir']}")
print("\nDone ✅")